# Milestone-1 Notebook

In [2]:
import pandas as pd
import numpy as np

In [3]:
train = pd.read_csv("../../data/train.csv")
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      2000 non-null   int64
 1   prompt  2000 non-null   str  
 2   A       2000 non-null   str  
 3   B       2000 non-null   str  
 4   C       2000 non-null   str  
 5   D       2000 non-null   str  
 6   E       2000 non-null   str  
 7   answer  2000 non-null   str  
dtypes: int64(1), str(7)
memory usage: 1.9 MB


## Q1. Calculate the frequency distribution of the correct answer (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?

In [4]:
freq = train['answer'].value_counts()

most_freq = freq.max()
least_freq = freq.min()

result = most_freq + least_freq

print(result)

814


## Q2. After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?

In [5]:
import string

vocab = set()

for text in train['prompt'].fillna(""):
    text = text.lower()

    # Remove punctuation
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    words = text.split()  # Split by whitespace

    vocab.update(words)

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 859


## Q3. Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?

In [6]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# row-1 prompt
prompt = train.loc[train['id']==1, 'prompt'].iloc[0]

print(prompt)

prompt = prompt.lower()

# Remove punctuation
prompt = prompt.translate(
    str.maketrans('', '', string.punctuation)
)

# Split into words
words = prompt.split()

print(words)

# Remove stop words
filtered_words = [
    word for word in words
    if word not in ENGLISH_STOP_WORDS
]

print("Filtered words:")
print(filtered_words)

print("\nNumber of words left:")
print(len(filtered_words))

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heideggers', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', 'among', 'the', 'listed', 'options']
Filtered words:
['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']

Number of words left:
13


## Q4. Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Combine prompt and all options into one text string per row
combined_text = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)
# print(combined_text)

vectorizer = TfidfVectorizer(stop_words="english")

X = vectorizer.fit_transform(combined_text)

print("Number of feature columns:", X.shape[1])

Number of feature columns: 2762


## Q5. Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

row = train.loc[train["id"] == 1].iloc[0]

prompt_vec = vectorizer.transform([row["prompt"]])
optionA_vec = vectorizer.transform([row["A"]])

similarity = cosine_similarity(
    prompt_vec,
    optionA_vec
)[0][0]

print(round(similarity, 4))

0.272


## Q6. Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options . Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.

In [9]:
options = ["A", "B", "C", "D", "E"]

correct = 0

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    scores = []

    for opt in options:

        option_vec = vectorizer.transform([row[opt]])

        sim = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        scores.append(sim)

    predicted = options[scores.index(max(scores))]

    if predicted == row["answer"]:
        correct += 1

accuracy = (correct / len(train)) * 100

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 13.55%


## Q7. If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?

Answer: 1.0

## Q8. If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?

Answer: 0.5

## Q9. The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [10]:
freq = train["answer"].value_counts()

top3 = freq.index[:3].tolist()

def apk(actual, predicted):
    if actual in predicted:
        return 1.0 / (predicted.index(actual) + 1)
    return 0.0

scores = [
    apk(ans, top3)
    for ans in train["answer"]
]

map3 = sum(scores) / len(scores)

print("Top 3 prediction:", top3)
print("MAP@3 =", round(map3, 6))

Top 3 prediction: ['B', 'C', 'A']
MAP@3 = 0.42125


## Q10. The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?

In [11]:
options = ["A", "B", "C", "D", "E"]

# MAP@3 function
def apk(actual, predicted, k=3):
    predicted = predicted[:k]

    if actual in predicted:
        return 1.0 / (predicted.index(actual) + 1)

    return 0.0

scores = []

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    similarities = []

    for opt in options:

        option_vec = vectorizer.transform([row[opt]])

        sim = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        similarities.append(sim)

    # Rank options by similarity
    ranked_indices = sorted(
        range(len(similarities)),
        key=lambda i: similarities[i],
        reverse=True
    )

    top3 = [options[i] for i in ranked_indices[:3]]

    score = apk(row["answer"], top3)

    scores.append(score)

map3 = sum(scores) / len(scores)

print("TF-IDF MAP@3 =", round(map3, 6))

TF-IDF MAP@3 = 0.296167
